# Sanity Check for taste_vector.npy

In [42]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/claraoberle/personal-book-recommender")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.build_k_nearest_neighbours import (
    get_eligible_books,
    score_candidate_knn,
    validate_knn,
    score_to_read,
)

EMBEDDINGD_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/embeddings.npy"
BOOK_ID_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/book_ids.npy"
TASTE_VECTOR_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/taste_vector.npy"
CLEAN_DF_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/embedding_join_clean_df.csv"

In [43]:
# Load data
embeddings = np.load(EMBEDDINGD_PATH, allow_pickle=True)
book_ids = np.load(BOOK_ID_PATH, allow_pickle=True)
taste_vector = np.load(TASTE_VECTOR_PATH, allow_pickle=True)

clean_df = pd.read_csv(CLEAN_DF_PATH)

clean_df["Book Id"] = clean_df["Book Id"].astype(str)

In [44]:
# Cosine similarity between taste vector and every book
similarities = embeddings @ taste_vector

# Create dataframe with results
similarity_df = pd.DataFrame({"Book Id": book_ids.astype(str), "similarity": similarities})

# Add book information
similarity_df = similarity_df.merge(clean_df[["Book Id", "Title", "My Rating", "Exclusive Shelf"]], on="Book Id", how="left")

## Top 10 books sorted by similarity

In [45]:
# Sort by similarity
top_10 = similarity_df.sort_values("similarity", ascending=False).head(10)

print(top_10[["Title", "similarity", "My Rating", "Exclusive Shelf"]].to_string(index=False))

                                                                  Title  similarity  My Rating   Exclusive Shelf
              Harry Potter and the Half-Blood Prince (Harry Potter, #6)    0.475732        5.0              read
           Harry Potter and the Order of the Phoenix (Harry Potter, #5)    0.469269        5.0              read
                Harry Potter and the Deathly Hallows (Harry Potter, #7)    0.412358        5.0              read
            Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)    0.409185        5.0              read
                                 Fire & Blood (A Targaryen History, #1)    0.405359        0.0           to-read
                  La comunidad del anillo (El señor de los anillos, #1)    0.384264        0.0 currently-reading
                 Harry Potter and the Goblet of Fire (Harry Potter, #4)    0.339458        5.0              read
             Harry Potter and the Chamber of Secrets (Harry Potter, #2)    0.319459        5.0  

In [46]:
print(f"Mean similarity: {similarities.mean():.3f}")
print(f"Min similarity: {similarities.min():.3f}")
print(f"Max similarity: {similarities.max():.3f}")

Mean similarity: -0.019
Min similarity: -0.388
Max similarity: 0.476


The similarity check produced encouraging results. The top-ranked books are dominated by highly rated Harry Potter books, with Fire & Blood ranking fifth with a similarity of 0.406 despite being on the to-read shelf and therefore being excluded from the construction of the taste vector. La comunidad del anillo also ranks highly despite being currently read and unrated, meaning it was likewise excluded from the taste vector. This suggests that the taste vector is capturing meaningful patterns in my reading preferences rather than simply reproducing the books used to construct it. However, the top 10 are heavily dominated by the Harry Potter series, which highlights a potential limitation of the approach: because the taste vector is an average of the weighted book embeddings, several books from the same tightly clustered series can reinforce the same region of the embedding space. As a result, the taste vector may be biased towards genres or series that I have read many books from, rather than representing all genres that I rate highly equally. To investigate this further, I will examine the similarity scores of the highest-ranked books and also inspect the lowest similarities to see how the taste vector distinguishes books that are less aligned with my preferences.

## High rated books

In [47]:
high_rated = similarity_df[similarity_df["My Rating"] >= 4].sort_values("similarity", ascending=False)

print(high_rated[["Title", "similarity", "My Rating"]].head(10).to_string(index=False))

                                                       Title  similarity  My Rating
   Harry Potter and the Half-Blood Prince (Harry Potter, #6)    0.475732        5.0
Harry Potter and the Order of the Phoenix (Harry Potter, #5)    0.469269        5.0
     Harry Potter and the Deathly Hallows (Harry Potter, #7)    0.412358        5.0
 Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)    0.409185        5.0
      Harry Potter and the Goblet of Fire (Harry Potter, #4)    0.339458        5.0
  Harry Potter and the Chamber of Secrets (Harry Potter, #2)    0.319459        5.0
                                 Archenemies (Renegades, #2)    0.280174        5.0
 Harry Potter and the Philosopher's Stone (Harry Potter, #1)    0.264268        5.0
           A Dance with Dragons (A Song of Ice and Fire, #5)    0.234550        5.0
                           Mockingjay (The Hunger Games, #3)    0.229457        5.0


Bottom of high rated books:

In [48]:
high_rated = similarity_df[similarity_df["My Rating"] >= 4].sort_values("similarity", ascending=True)

print(high_rated[["Title", "similarity", "My Rating"]].head(10).to_string(index=False))

                                                                    Title  similarity  My Rating
                                                        Love on the Brain   -0.278132        4.0
                                              The Roommate (Shameless #1)   -0.241500        4.0
            P.S. I Still Love You (To All the Boys I've Loved Before, #2)   -0.222047        4.0
                                        El amor en los tiempos del cólera   -0.207248        4.0
                                                          The Nightingale   -0.193039        4.0
                          Get a Life, Chloe Brown (The Brown Sisters, #1)   -0.190286        4.0
                                                      Pride and Prejudice   -0.163815        4.0
                                          Six of Crows (Six of Crows, #1)   -0.161110        5.0
                                                  Una segunda oportunidad   -0.151830        4.0
To All the Boys I've Loved Bef

The bottom of the list of highly rated books reveals an important limitation of the single taste vector. Every book in this group was rated 4 or 5 stars, yet all of them have negative similarity with the taste vector. The pattern is also meaningful: many are contemporary romance or literary/historical fiction, such as Love on the Brain, The Roommate, Pride and Prejudice, and The Nightingale. Even Six of Crows, a fantasy title rated 5 stars, has a negative similarity, suggesting that the embedding space distinguishes between different types of fantasy rather than treating them as one broad preference.

This is explained by the fact that the mean similarity across the library is approximately -0.019, which is essentially zero. Averaging embeddings from books belonging to different preference clusters can cause their directions to partially cancel each other out. In this case, the taste vector appears to be dominated by the tightly clustered fantasy books, while other genres that I also rate highly point in different directions and therefore become poorly represented by the single average vector. This is a useful finding rather than simply a failure of the approach: it suggests that my preferences are better represented by multiple clusters or separate taste vectors than by a single averaged vector, which supports experimenting with alternative representations of preferences in the next stage of the recommender.

## K-nearest neighbour

In [49]:
def inspect_book_knn(book_id, clean_df, eligible_vectors, eligible_ratings, eligible_ids, mean_rating, embeddings, book_ids, k=5):
    """Inspect the k nearest eligible neighbours for a given book."""

    book_id = str(book_id)

    # Find the book in the embedding array
    embedding_indices = np.where(book_ids.astype(str) == book_id)[0]

    if len(embedding_indices) == 0:
        raise ValueError(f"Book Id {book_id} not found in book_ids.")

    book_vector = embeddings[embedding_indices[0]]

    # Find the book's information
    book_info = clean_df[clean_df["Book Id"] == book_id]

    if book_info.empty:
        raise ValueError(f"Book Id {book_id} not found in clean_df.")

    # Calculate similarity to all eligible books
    similarities = eligible_vectors @ book_vector

    # Exclude the book itself
    mask = eligible_ids != book_id

    similarities = similarities[mask]
    ratings = eligible_ratings[mask]
    ids = eligible_ids[mask]

    # Get k most similar books
    top_k_idx = similarities.argsort()[::-1][:k]

    # Create result dataframe
    neighbors = pd.DataFrame({
        "Book Id": ids[top_k_idx],
        "similarity": similarities[top_k_idx],
        "My Rating": ratings[top_k_idx]
    })

    # Add titles
    neighbors = neighbors.merge(
        clean_df[["Book Id", "Title"]],
        on="Book Id",
        how="left"
    )

    # Calculate predicted rating
    predicted_deviation = (neighbors["My Rating"] - mean_rating).mean()

    predicted_rating = mean_rating + predicted_deviation

    print(f"Book: {book_info.iloc[0]['Title']}")
    print(f"Book Id: {book_id}")
    print(f"Actual rating: {book_info.iloc[0]['My Rating']}")
    print(f"Mean rating: {mean_rating:.3f}")
    print(f"Predicted rating: {predicted_rating:.3f}")
    print("\nNearest neighbours:")

    print(neighbors[["Title", "similarity", "My Rating"]].to_string(index=False))

    return neighbors

In [50]:
eligible_vectors, eligible_ratings, eligible_ids, mean_rating = get_eligible_books(clean_df, embeddings, book_ids)

### Trying different (random) books

In [51]:
harry_potter_6_id = "1"
archenemies_id = "35425827"
six_of_crows_id = "23437156"
# to all the boys i have loved before
tatbihlb_id = "15749186"
pride_prejudice_id = "1885"
the_nightingale_id = "21853621"

In [52]:
neighbors = inspect_book_knn(harry_potter_6_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: Harry Potter and the Half-Blood Prince (Harry Potter, #6)
Book Id: 1
Actual rating: 5.0
Mean rating: 3.897
Predicted rating: 4.600

Nearest neighbours:
                                                                  Title  similarity  My Rating
           Harry Potter and the Order of the Phoenix (Harry Potter, #5)    0.690136        5.0
                Harry Potter and the Deathly Hallows (Harry Potter, #7)    0.664030        5.0
            Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)    0.633990        5.0
Harry Potter and the Cursed Child: Parts One and Two (Harry Potter, #8)    0.578720        3.0
                 Harry Potter and the Goblet of Fire (Harry Potter, #4)    0.563999        5.0


In [53]:
neighbors = inspect_book_knn(archenemies_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: Archenemies (Renegades, #2)
Book Id: 35425827
Actual rating: 5.0
Mean rating: 3.897
Predicted rating: 4.200

Nearest neighbours:
                                            Title  similarity  My Rating
                        Supernova (Renegades, #3)    0.681522        5.0
                        Renegades (Renegades, #1)    0.592739        5.0
Siege and Storm (The Shadow and Bone Trilogy, #2)    0.540911        3.0
            Shadow and Bone (Shadow and Bone, #1)    0.537458        4.0
                             All You Need Is Kill    0.504412        4.0


In [54]:
neighbors = inspect_book_knn(six_of_crows_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: Six of Crows (Six of Crows, #1)
Book Id: 23437156
Actual rating: 5.0
Mean rating: 3.897
Predicted rating: 3.400

Nearest neighbours:
                                         Title  similarity  My Rating
         Ruin and Rising (Shadow and Bone, #3)    0.980788        4.0
                                      The Help    0.606038        3.0
Todo lo que nunca fuimos (Deja que ocurra, #1)    0.599765        3.0
                           Pride and Prejudice    0.587568        4.0
                              Heist (Darks #1)    0.573541        3.0


In [55]:
neighbors = inspect_book_knn(tatbihlb_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: To All the Boys I've Loved Before (To All the Boys I've Loved Before, #1)
Book Id: 15749186
Actual rating: 4.0
Mean rating: 3.897
Predicted rating: 3.800

Nearest neighbours:
                                                                Title  similarity  My Rating
Always and Forever, Lara Jean (To All the Boys I've Loved Before, #3)    0.883541        3.0
            Dune (edición especial película) (Las crónicas de Dune 1)    0.741512        5.0
        P.S. I Still Love You (To All the Boys I've Loved Before, #2)    0.721945        4.0
                                                Caraval (Caraval, #1)    0.663960        2.0
                                                   The Secret History    0.621013        5.0


In [56]:
neighbors = inspect_book_knn(pride_prejudice_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)

Book: Pride and Prejudice
Book Id: 1885
Actual rating: 4.0
Mean rating: 3.897
Predicted rating: 3.200

Nearest neighbours:
                                          Title  similarity  My Rating
 Todo lo que nunca fuimos (Deja que ocurra, #1)    0.663788        3.0
                     Margo's Got Money Troubles    0.622791        3.0
Call Me By Your Name (Call Me By Your Name, #1)    0.595939        3.0
                            Love, Theoretically    0.595829        3.0
          Ruin and Rising (Shadow and Bone, #3)    0.595540        4.0


In [57]:
neighbors = inspect_book_knn(the_nightingale_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: The Nightingale
Book Id: 21853621
Actual rating: 4.0
Mean rating: 3.897
Predicted rating: 3.200

Nearest neighbours:
                                     Title  similarity  My Rating
                          Mientras vivimos    0.663526        3.0
         Me Before You (Me Before You, #1)    0.655662        3.0
Los pilares de la tierra (Kingsbridge, #1)    0.612283        4.0
                       The Love Hypothesis    0.608644        3.0
                Margo's Got Money Troubles    0.558083        3.0


### Validate knn

In [58]:
knn_results = validate_knn(eligible_vectors, eligible_ratings, eligible_ids, mean_rating)

knn_results = knn_results.merge(
        clean_df[["Book Id", "Title"]],
        on="Book Id",
        how="left"
    )

knn_results.head()

,Book Id,predicted,actual,Title
0,55659629,-0.497059,-1.897059,"It Happened One Summer (Bellinger Sisters, #1)"
1,54438718,0.102941,0.102941,Una promesa de juventud
2,22628,0.702941,1.102941,The Perks of Being a Wallflower
3,36336078,-0.897059,-0.897059,"Call Me By Your Name (Call Me By Your Name, #1)"
4,213243908,0.102941,-1.897059,"First-Time Caller (Heartstrings, #1)"


In [59]:
correlation_knn = knn_results["predicted"].corr(knn_results["actual"])

print(correlation_knn)

0.2523727046190534


In [60]:
first_time_caller_id = 213243908
neighbors = inspect_book_knn(first_time_caller_id, clean_df, eligible_vectors, eligible_ratings, eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: First-Time Caller (Heartstrings, #1)
Book Id: 213243908
Actual rating: 2.0
Mean rating: 3.897
Predicted rating: 4.000

Nearest neighbours:
                                          Title  similarity  My Rating
                            Nosotros en la Luna    0.512011        5.0
                              Love on the Brain    0.482065        4.0
                            Love, Theoretically    0.452136        3.0
                    The Roommate (Shameless #1)    0.444078        4.0
Get a Life, Chloe Brown (The Brown Sisters, #1)    0.436657        4.0


## Comparing taste vector and K nearest neighbors

In [61]:
def validate_centroid(eligible_vectors, eligible_ratings, eligible_ids, mean_rating, taste_vector):
    results = []

    for i, book_id in enumerate(eligible_ids):
        similarity = eligible_vectors[i] @ taste_vector

        actual_deviation = eligible_ratings[i] - mean_rating
        results.append({
            "Book Id": book_id,
            "predicted": similarity,
            "actual": actual_deviation
        })

    return pd.DataFrame(results)

In [62]:
centroid_results = validate_centroid(eligible_vectors, eligible_ratings, eligible_ids, mean_rating, taste_vector)

centroid_corr = centroid_results["predicted"].corr(centroid_results["actual"])

print(f"Centroid correlation: {centroid_corr:.3f}")

Centroid correlation: 0.576


The **centroid method**, which I previously called the **taste vector approach**, represents my overall reading preferences with a single vector. This taste vector is created from the embeddings of the books I have rated, giving more importance to books that I rated above or below my average rating. To predict how well I might like a new book, its embedding is compared with this taste vector using cosine similarity. The correlation results are 0.576 for the centroid method and 0.252 for k-NN. However, this does not mean that the centroid method is always better. Earlier, we saw that the centroid method gave negative similarity to books such as *Pride and Prejudice*, *The Nightingale*, and *Love on the Brain*, even though they are books I rated 4 stars. This can happen because most of my 165 books belong to a similar fantasy/Harry Potter-type group. Since these books make up a large part of my data and also helped create the taste vector, they can have a strong effect on the overall correlation. This means that the single correlation number can hide problems in smaller groups, such as romance or literary fiction. Therefore, it is important to look at specific examples and different groups of books, rather than only relying on the overall correlation.

To investigate this further, I will focus specifically on the books where the centroid method predicts a negative similarity. These are the books where we already know that the taste vector can fail, so comparing both methods on this subset will show whether k-NN can recover information that the centroid method misses.

In [75]:
def correlation_comparison(knn_results, centroid_results, positive=False):
    comparison = knn_results.copy()
    comparison["centroid_predicted"] = centroid_results["predicted"]

    if positive:
        subset = comparison[comparison["centroid_predicted"] >= 0]
        label = "positive"
    else:
        subset = comparison[comparison["centroid_predicted"] < 0]
        label = "negative"

    print(f"Number of {label}-centroid books: {len(subset)}")

    knn_corr = subset["predicted"].corr(subset["actual"])
    centroid_corr = subset["centroid_predicted"].corr(subset["actual"])

    print(f"KNN correlation:      {knn_corr:.3f}")
    print(f"Centroid correlation: {centroid_corr:.3f}")

In [76]:
correlation_comparison(knn_results, centroid_results)

Number of negative-centroid books: 76
KNN correlation:      0.019
Centroid correlation: 0.416


### Leave-one-out for centroid

The first results did not give us the expected outcome: the centroid (taste vector) method had a higher correlation than k-NN, even though we had already found examples where the taste vector clearly failed. One possible reason is that the original taste vector was built using **all the books**, including the book we were trying to predict. This gives the centroid method information about the answer before making the prediction, making the comparison unfair. To fix this, we use **Leave-One-Out (LOO)** validation. The idea is simple: for each book, we temporarily remove it from the data, build the taste vector again without that book, and then use this new vector to predict the rating of the removed book. We repeat this for every book. This way, the centroid method never uses the book it is trying to predict, just like our k-NN method already does, giving us a fair comparison between the two methods.

In [65]:
# Calculate each book's weight
weights = eligible_ratings - mean_rating

# Weighted sum used to create the original taste vector
weighted_sum = np.sum(
    weights[:, np.newaxis] * eligible_vectors,
    axis=0
)

# Total absolute weight
total_abs_weight = np.sum(np.abs(weights))

In [66]:
def validate_centroid_loo(
    eligible_vectors,
    eligible_ratings,
    eligible_ids,
    mean_rating
):
    results = []

    # Build the full weighted sum once
    weights = eligible_ratings - mean_rating

    weighted_sum = np.sum(
        weights[:, np.newaxis] * eligible_vectors,
        axis=0
    )

    total_abs_weight = np.sum(np.abs(weights))

    for i, book_id in enumerate(eligible_ids):

        # Remove this book's contribution
        adjusted_sum = weighted_sum - (
            weights[i] * eligible_vectors[i]
        )

        adjusted_total_weight = (
            total_abs_weight - abs(weights[i])
        )

        # Avoid division by zero
        if adjusted_total_weight == 0:
            predicted = np.nan
        else:
            # Rebuild the taste vector without this book
            loo_taste_vector = adjusted_sum / adjusted_total_weight

            # Normalize it
            norm = np.linalg.norm(loo_taste_vector)

            if norm == 0:
                predicted = np.nan
            else:
                loo_taste_vector = loo_taste_vector / norm

                # Predict this book using the LOO taste vector
                predicted = (
                    eligible_vectors[i] @ loo_taste_vector
                )

        actual = eligible_ratings[i] - mean_rating

        results.append({
            "Book Id": book_id,
            "predicted": predicted,
            "actual": actual
        })

    return pd.DataFrame(results)

In [67]:
centroid_results_loo = validate_centroid_loo(eligible_vectors, eligible_ratings, eligible_ids, mean_rating)

In [72]:
centroid_corr_loo = centroid_results_loo["predicted"].corr(centroid_results_loo["actual"])

print(f"Centroid LOO correlation: {centroid_corr_loo:.3f}")

Centroid LOO correlation: 0.134


In [77]:
correlation_comparison(knn_results, centroid_results_loo)

Number of negative-centroid books: 120
KNN correlation:      0.216
Centroid correlation: 0.021


In [78]:
correlation_comparison(knn_results, centroid_results_loo, positive=True)

Number of positive-centroid books: 7
KNN correlation:      -1.000
Centroid correlation: 0.259


After removing the data leakage, the results are much more realistic. The centroid method's overall correlation falls from 0.576 to 0.134, which means it is only slightly better than having no relationship with the actual ratings. In the difficult subset, the centroid correlation is almost zero (0.021), while k-NN has a positive correlation of 0.216. This is an important result because it shows that **k-NN performs better specifically in the area where the centroid method struggles**, and it does so without using information from the book being predicted. This also confirms the problems we found earlier with books such as *Pride and Prejudice*, *The Nightingale*, and *Love on the Brain*, but now the conclusion is supported by a proper validation method rather than just a few examples. However, the correlations are still quite low, so neither method is a strong predictor of my ratings. This is not necessarily a failure, but reflects the limitations of the project: there are only around 165 books, my reading preferences contain different groups, the book embeddings cannot capture things like writing quality, and k-NN can sometimes be forced to use weak matches when there are not enough similar books. Overall, the results suggest that k-NN is a reasonable and defensible content-based recommender for this dataset, but it should not be expected to predict my ratings with high accuracy.


## score_to_read